# 01 — Data Exploration

Explore the HAM10000 dataset: class distribution, sample images, and key statistics.

**Run this notebook first** to understand the data before training.

In [ ]:
# --- Setup (run this cell first) ---
import sys, os
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    !pip install -q torch torchvision pandas matplotlib seaborn Pillow

    # Download dataset from Kaggle
    !pip install -q kaggle
    from google.colab import files
    if not os.path.exists(os.path.expanduser('~/.kaggle/kaggle.json')):
        print("Upload your kaggle.json file:")
        uploaded = files.upload()
        !mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json

    !kaggle datasets download -d kmader/skin-cancer-mnist-ham10000 -p /content/data/ --quiet
    !unzip -q -o /content/data/skin-cancer-mnist-ham10000.zip -d /content/data/HAM10000/
    DATA_DIR = '/content/data/HAM10000'
else:
    DATA_DIR = '../data/HAM10000'

print(f'Data directory: {DATA_DIR}')

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from PIL import Image
import numpy as np

# Load metadata
metadata = pd.read_csv(Path(DATA_DIR) / 'HAM10000_metadata.csv')
print(f'Total images: {len(metadata)}')
print(f'Unique lesions: {metadata["lesion_id"].nunique()}')
print(f'\nColumns: {list(metadata.columns)}')
metadata.head()

In [ ]:
# --- Class Distribution ---
class_full_names = {
    'akiec': 'Actinic Keratoses',
    'bcc': 'Basal Cell Carcinoma',
    'bkl': 'Benign Keratosis',
    'df': 'Dermatofibroma',
    'mel': 'Melanoma',
    'nv': 'Melanocytic Nevi',
    'vasc': 'Vascular Lesions',
}

fig, ax = plt.subplots(figsize=(10, 5))
counts = metadata['dx'].value_counts().sort_index()
bars = ax.bar(counts.index, counts.values, color=sns.color_palette('Set2', 7))
ax.set_xlabel('Diagnosis')
ax.set_ylabel('Count')
ax.set_title('HAM10000 Class Distribution (Severely Imbalanced)')
for bar, count in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
            f'{count}\n({count/len(metadata)*100:.1f}%)', ha='center', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# --- Sample Images (one per class) ---
fig, axes = plt.subplots(1, 7, figsize=(21, 3))

for idx, (dx, name) in enumerate(sorted(class_full_names.items())):
    sample = metadata[metadata['dx'] == dx].iloc[0]
    img_path = Path(DATA_DIR) / 'images' / f"{sample['image_id']}.jpg"
    if img_path.exists():
        img = Image.open(img_path)
        axes[idx].imshow(img)
    axes[idx].set_title(f"{dx}\n{name}", fontsize=9)
    axes[idx].axis('off')

plt.suptitle('Sample Images — One Per Class', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# --- Metadata Distributions ---
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Age distribution
metadata['age'].dropna().hist(bins=20, ax=axes[0], color='steelblue')
axes[0].set_title('Age Distribution')
axes[0].set_xlabel('Age')

# Sex distribution
metadata['sex'].value_counts().plot.bar(ax=axes[1], color=['steelblue', 'coral'])
axes[1].set_title('Sex Distribution')

# Localization
metadata['localization'].value_counts().head(10).plot.barh(ax=axes[2], color='steelblue')
axes[2].set_title('Top 10 Localizations')

plt.tight_layout()
plt.show()

In [ ]:
# --- Key Statistics Summary ---
print('=== HAM10000 Dataset Summary ===')
print(f'Total images: {len(metadata)}')
print(f'Unique lesions: {metadata["lesion_id"].nunique()}')
print(f'Images with duplicate lesions: {len(metadata) - metadata["lesion_id"].nunique()}')
print(f'\nClass imbalance ratio (max/min): {counts.max()}/{counts.min()} = {counts.max()/counts.min():.1f}x')
print(f'\nDiagnosis confirmation methods:')
print(metadata['dx_type'].value_counts().to_string())
print(f'\nMissing values:')
print(metadata.isnull().sum()[metadata.isnull().sum() > 0].to_string())